Estimation de l'effet causal de la fermeture des établissement 'social' sur le vote RN à l'aide d'un staggered DID

Import des bibliothèques

In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from did_multiplegt_dyn import DidMultiplegtDyn  

Import des données

In [25]:
# Social
df_rnp = pd.read_csv(r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\Partie Econometrie\communes_social_rnp.csv", index_col=0)
df_rnp['codecommune'] = df_rnp['codecommune'].astype(str).str.zfill(5)

df_rp = pd.read_csv(r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\Partie Econometrie\communes_social_rp.csv", index_col=0)
df_rp['codecommune'] = df_rp['codecommune'].astype(str).str.zfill(5)

df_ui = pd.read_csv(r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\Partie Econometrie\communes_social_ui.csv", index_col=0)
df_ui['codecommune'] = df_ui['codecommune'].astype(str).str.zfill(5)

df_ud = pd.read_csv(r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\Partie Econometrie\communes_social_d.csv", index_col=0)
df_ud['codecommune'] = df_ud['codecommune'].astype(str).str.zfill(5)

C:\Users\yancr\AppData\Local\Temp\ipykernel_30400\2641705729.py:2: DtypeWarning: Columns (0: Annee) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rnp = pd.read_csv(r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\Partie Econometrie\communes_social_rnp.csv", index_col=0)
C:\Users\yancr\AppData\Local\Temp\ipykernel_30400\2641705729.py:5: DtypeWarning: Columns (0: Annee) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rp = pd.read_csv(r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\Partie Econometrie\communes_social_rp.csv", index_col=0)
C:\Users\yancr\AppData\Local\Temp\ipykernel_30400\2641705729.py:8: DtypeWarning: Columns (0: Annee) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ui = pd.read_csv(r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\Partie Econometrie\communes_social_ui.csv", index_col=0)


Fonction d'attribution du traitement

In [42]:
def traitement(df):
    # 1. On définit notre seuil de traitement à partir de l'année 2002
    df_2002 = df[df['Annee'] == 2002].copy()
    df_2002['seuil'] = df_2002['total_equipements'] * 0.5

    # 2. On renseigne ce seuil dans le DataFrame pour comparer les niveaux annuels
    mapping_seuil = df_2002.set_index('codecommune')['seuil']
    df['seuil'] = df['codecommune'].map(mapping_seuil)

    # 3. Première passe : condition simple (1 si on passe sous le seuil, 0 sinon)
    df['traitée'] = (df['total_equipements'] < df['seuil']).astype(int)

    # 4. ÉTAPE CRUCIALE : Trier le tableau pour garantir l'ordre chronologique
    df = df.sort_values(by=['codecommune', 'Annee'])

    # 5. L'EFFET CLIQUET : On applique un maximum cumulé par commune.
    # Dès qu'une commune obtient un 1, toutes les lignes suivantes pour cette commune vaudront 1.
    df['traitée'] = df.groupby('codecommune')['traitée'].cummax()

    return df

## Calcul du traitement

### RNP

In [43]:
df_rnp = traitement(df_rnp)

### RP

In [44]:
df_rp = traitement(df_rp)

### UI

In [45]:
df_ui = traitement(df_ui)

### UD

In [46]:
df_ud = traitement(df_ud)

## DID

Definition du modele

In [47]:
def DID (df) :
    # 1. On filtre la base pour ne garder que les années d'élection présidentielle
    annees_pres = [2002, 2007, 2012, 2017, 2022]
    df_pres = df[df['Annee'].isin(annees_pres)].copy()

    # 2. Nettoyage : on retire les lignes avec des valeurs manquantes essentielles
    df_pres = df_pres.dropna(subset=['vote_RN_pres', 'traitée'])

    # 3. On force les types en formats "simples" (Numpy) pour faciliter la conversion
    # (Cela vous évite l'erreur PyArrow)
    df_pres['vote_RN_pres'] = df_pres['vote_RN_pres'].astype('float64')
    df_pres['traitée'] = df_pres['traitée'].astype('int64')
    df_pres['Annee'] = df_pres['Annee'].astype('int64')
    # Note : S'il y a un souci avec 'codecommune' (par exemple si c'est un format 'object'),
    # vous pouvez aussi le forcer en string classique : df_pres['codecommune'] = df_pres['codecommune'].astype('str')

    # 4. Conversion en Polars
    df_pres_polars = pl.from_pandas(df_pres)

    # 5. Estimation Staggered DiD (avec le nom de classe corrigé)
    modele_did = DidMultiplegtDyn(
        df=df_pres_polars,         
        outcome='vote_RN_pres',    # Variable dépendante
        group='codecommune',       # Identifiant du groupe
        time='Annee',              # Variable temporelle
        treatment='traitée',       # Indicateur de traitement
        placebo=1,                 # Tester 1 période avant le traitement (pre-trends)
        effects=2,                 # Estimer l'effet sur les 2 périodes suivant le traitement
        cluster='codecommune'      # Clustering des erreurs-types
    )

    modele_did.fit()

    return modele_did.summary()

### RNP

In [48]:
DID(df_rnp)

             Estimation of treatment effects: Event-study effects
               Block  Estimate       SE     LB CI    UB CI       N  Switchers     N.w  Switchers.w
            Effect_1  0.002427 0.000898  0.000667 0.004187 42695.0     3135.0 42695.0       3135.0
            Effect_2  0.006159 0.001295  0.003621 0.008696 31002.0     2509.0 31002.0       2509.0
Average_Total_Effect  0.004086 0.000949  0.002226 0.005946 45212.0     5644.0 45212.0       5644.0
           Placebo_1 -0.000624 0.001045 -0.002672 0.001424 30682.0     2197.0 30682.0       2197.0
Test of joint nullity of the effects: p-value = 0.000012

The development of this package was funded by the European Union.
ERC REALLYCREDIBLE - GA N. 101043899


,Block,Estimate,SE,LB CI,UB CI,N,Switchers,N.w,Switchers.w
0,Effect_1,0.002427,0.000898,0.000667,0.004187,42695.0,3135.0,42695.0,3135.0
1,Effect_2,0.006159,0.001295,0.003621,0.008696,31002.0,2509.0,31002.0,2509.0
2,Average_Total_Effect,0.004086,0.000949,0.002226,0.005946,45212.0,5644.0,45212.0,5644.0
3,Placebo_1,-0.000624,0.001045,-0.002672,0.001424,30682.0,2197.0,30682.0,2197.0


### RP

In [49]:
DID(df_rp)

             Estimation of treatment effects: Event-study effects
               Block  Estimate       SE     LB CI    UB CI       N  Switchers     N.w  Switchers.w
            Effect_1  0.003082 0.000805  0.001504 0.004661 39612.0     3022.0 39612.0       3022.0
            Effect_2  0.006455 0.001150  0.004201 0.008710 28781.0     2483.0 28781.0       2483.0
Average_Total_Effect  0.004604 0.000869  0.002900 0.006307 42098.0     5505.0 42098.0       5505.0
           Placebo_1 -0.001745 0.000970 -0.003646 0.000157 28380.0     2085.0 28380.0       2085.0
Test of joint nullity of the effects: p-value = 0.000000

The development of this package was funded by the European Union.
ERC REALLYCREDIBLE - GA N. 101043899


,Block,Estimate,SE,LB CI,UB CI,N,Switchers,N.w,Switchers.w
0,Effect_1,0.003082,0.000805,0.001504,0.004661,39612.0,3022.0,39612.0,3022.0
1,Effect_2,0.006455,0.001150,0.004201,0.008710,28781.0,2483.0,28781.0,2483.0
2,Average_Total_Effect,0.004604,0.000869,0.002900,0.006307,42098.0,5505.0,42098.0,5505.0
3,Placebo_1,-0.001745,0.000970,-0.003646,0.000157,28380.0,2085.0,28380.0,2085.0


### UI

In [52]:
DID(df_ui)

             Estimation of treatment effects: Event-study effects
               Block  Estimate       SE     LB CI     UB CI       N  Switchers     N.w  Switchers.w
            Effect_1  0.008753 0.001427  0.005956  0.011549 12931.0      561.0 12931.0        561.0
            Effect_2  0.018021 0.002346  0.013422  0.022620  9504.0      454.0  9504.0        454.0
Average_Total_Effect  0.012898 0.001685  0.009595  0.016201 13385.0     1015.0 13385.0       1015.0
           Placebo_1 -0.006674 0.001785 -0.010173 -0.003176  9471.0      421.0  9471.0        421.0
Test of joint nullity of the effects: p-value = 0.000000

The development of this package was funded by the European Union.
ERC REALLYCREDIBLE - GA N. 101043899


,Block,Estimate,SE,LB CI,UB CI,N,Switchers,N.w,Switchers.w
0,Effect_1,0.008753,0.001427,0.005956,0.011549,12931.0,561.0,12931.0,561.0
1,Effect_2,0.018021,0.002346,0.013422,0.022620,9504.0,454.0,9504.0,454.0
2,Average_Total_Effect,0.012898,0.001685,0.009595,0.016201,13385.0,1015.0,13385.0,1015.0
3,Placebo_1,-0.006674,0.001785,-0.010173,-0.003176,9471.0,421.0,9471.0,421.0


### UD

In [53]:
DID(df_ud)

             Estimation of treatment effects: Event-study effects
               Block  Estimate       SE     LB CI     UB CI      N  Switchers    N.w  Switchers.w
            Effect_1  0.023489 0.005125  0.013443  0.033534 2791.0       33.0 2791.0         33.0
            Effect_2  0.040334 0.008622  0.023435  0.057234 2081.0       26.0 2081.0         26.0
Average_Total_Effect  0.030912 0.006086  0.018984  0.042841 2818.0       59.0 2818.0         59.0
           Placebo_1 -0.023913 0.005708 -0.035100 -0.012726 2079.0       25.0 2079.0         25.0
Test of joint nullity of the effects: p-value = 0.000002

The development of this package was funded by the European Union.
ERC REALLYCREDIBLE - GA N. 101043899


,Block,Estimate,SE,LB CI,UB CI,N,Switchers,N.w,Switchers.w
0,Effect_1,0.023489,0.005125,0.013443,0.033534,2791.0,33.0,2791.0,33.0
1,Effect_2,0.040334,0.008622,0.023435,0.057234,2081.0,26.0,2081.0,26.0
2,Average_Total_Effect,0.030912,0.006086,0.018984,0.042841,2818.0,59.0,2818.0,59.0
3,Placebo_1,-0.023913,0.005708,-0.035100,-0.012726,2079.0,25.0,2079.0,25.0
